In [ ]:
%matplotlib ipympl
from pathlib import Path

from matplotlib import pyplot as plt
from ipywidgets import widgets

from topepan import plot as tpp

In [ ]:
# The CrunchTope input deck. Its folder holds the .tec output and the deck itself carries the
# output times, so this one path gives both. Edit it and re-run from here down.
DECK = '/path/to/your/crunchtope/run/model.in'

catList, max_time = tpp.data_cats(Path(DECK).parent)

# Read from the deck rather than from a .tec header: MineralPercent, which the alternative reads,
# only exists in CrunchTope 2.10 and later, so a run from an older build has no such file.
times = tpp.read_times(DECK)

In [ ]:
file_cat = widgets.ToggleButtons(options=catList, description='Output type')
time_slider = widgets.IntSlider(value=1, min=1, max=max_time, step=1, description='Time step')
plot_var = widgets.Select(options=tpp.read_tecplot(file_cat.value, time_slider.value)[1][3:],
                          description='Variable')
# X down the y axis is the depth convention; X across the x axis reads better for a flow path.
orientation = widgets.ToggleButtons(
    options=[('X on y axis (depth)', True), ('X on x axis', False)],
    value=True, description='Orientation')
log_scale = widgets.Checkbox(value=False, description='log10')


def update_plot_vars(*args):
    plot_var.options = tpp.read_tecplot(file_cat.value, time_slider.value)[1][3:]


def update_plot(time, file_cat, plot_var, orientation, log10):
    if plot_var is None:
        return
    df, column_headers = tpp.read_tecplot(file_cat, time)
    # Ranged over every timestep, so the axis does not jump about as the slider moves.
    lower, upper = tpp.plot_var_range(time_slider.max, file_cat, plot_var)

    # Pad the value axis away from the data, and give a flat profile a range it can be seen in.
    if upper > 0:
        upper = upper * 1.02
    elif upper < 0:
        upper = upper * 0.98
    else:
        upper = -lower

    if lower > 0:
        lower = lower * 0.98
    elif lower < 0:
        lower = lower * 1.02
    else:
        lower = -upper

    if upper == lower:
        upper = upper + 1
        lower = lower - 1

    tpp.draw_profile(ax, df['X'], df[plot_var], vertical=orientation, lower=lower, upper=upper,
                     value_label=plot_var, log10=log10)
    # Which snapshot is on screen, since the slider only gives its index. A title rather than an
    # axis label because this is a browser, not a figure headed for a paper.
    if time - 1 < len(times):
        ax.set_title(f'time = {times[time - 1]:g}')
    fig.canvas.draw_idle()


file_cat.observe(update_plot_vars, 'value')
fig, ax, line = tpp.initialise1D(sorted(catList)[0])
widgets.interact(update_plot, file_cat=file_cat, time=time_slider, plot_var=plot_var,
                 orientation=orientation, log10=log_scale)

In [ ]:
plt.close('all')